# Imports

In [1]:
from google.colab import files
import io
import os

import numpy as np
import pandas as pd

from sklearn.utils import shuffle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from keras.callbacks import EarlyStopping

# Dataset

Load the pre-processed dataset.

In [2]:
# Remove file from colab environment if it already exists, to prevent multiple uploads when running code cell
!rm -rf HDB_Resale_Train.csv
!rm -rf HDB_Resale_Test.csv

# Create dictionary and store the data files
uploaded = files.upload()

# List all files and directories in the current working directory
print(os.listdir('.'))

# List files in uploaded
print(uploaded.keys())

# Train and test set
Train_df = pd.read_csv(io.BytesIO(uploaded['HDB_Resale_Train.csv']))
Test_df = pd.read_csv(io.BytesIO(uploaded['HDB_Resale_Test.csv']))

Saving HDB_Resale_Train.csv to HDB_Resale_Train.csv
Saving HDB_Resale_Test.csv to HDB_Resale_Test.csv
['.config', 'HDB_Resale_Train.csv', 'HDB_Resale_Test.csv', 'sample_data']
dict_keys(['HDB_Resale_Train.csv', 'HDB_Resale_Test.csv'])


### Manually split dataset into regions

We will use the values of the feature 'town' to group the data into 5 main regions of Singapore: North, South, East, West and Central. Then perform one hot encoding and scaling on the new datasets.

In [3]:
# Get a list of the unique towns in training and testing set
# to ensure all values are accounted for
print(Train_df['town'].unique())
print(Test_df['town'].unique())

['ANG MO KIO' 'BEDOK' 'BISHAN' 'BUKIT BATOK' 'BUKIT MERAH' 'BUKIT PANJANG'
 'BUKIT TIMAH' 'CENTRAL AREA' 'CHOA CHU KANG' 'CLEMENTI' 'GEYLANG'
 'HOUGANG' 'JURONG EAST' 'JURONG WEST' 'KALLANG/WHAMPOA' 'MARINE PARADE'
 'PASIR RIS' 'PUNGGOL' 'QUEENSTOWN' 'SEMBAWANG' 'SENGKANG' 'SERANGOON'
 'TAMPINES' 'TOA PAYOH' 'WOODLANDS' 'YISHUN']
['ANG MO KIO' 'BEDOK' 'BISHAN' 'BUKIT BATOK' 'BUKIT MERAH' 'BUKIT PANJANG'
 'BUKIT TIMAH' 'CENTRAL AREA' 'CHOA CHU KANG' 'CLEMENTI' 'GEYLANG'
 'HOUGANG' 'JURONG EAST' 'JURONG WEST' 'KALLANG/WHAMPOA' 'MARINE PARADE'
 'PASIR RIS' 'PUNGGOL' 'QUEENSTOWN' 'SEMBAWANG' 'SENGKANG' 'SERANGOON'
 'TAMPINES' 'TOA PAYOH' 'WOODLANDS' 'YISHUN']


In [4]:
# Define regions and their corresponding towns
regions = {
    'North': ['PUNGGOL', 'SEMBAWANG', 'SENGKANG', 'WOODLANDS', 'YISHUN'],
    'South': ['BUKIT MERAH', 'GEYLANG', 'MARINE PARADE', 'QUEENSTOWN'],
    'East': ['BEDOK', 'HOUGANG', 'PASIR RIS', 'TAMPINES'],
    'West': ['BUKIT BATOK', 'BUKIT PANJANG', 'CHOA CHU KANG', 'CLEMENTI', 'JURONG EAST', 'JURONG WEST'],
    'Central': ['ANG MO KIO', 'BISHAN', 'BUKIT TIMAH', 'CENTRAL AREA', 'KALLANG/WHAMPOA', 'SERANGOON', 'TOA PAYOH']
}

# Initialise dictionaries to store the split dataframes
regional_train_dfs = {}
regional_test_dfs = {}

# Loop through regions to split the datasets
for region_name, towns in regions.items():
    regional_train_dfs[region_name] = Train_df[Train_df['town'].isin(towns)]
    regional_test_dfs[region_name] = Test_df[Test_df['town'].isin(towns)]


for region_name in regions.keys():
    # Shuffle the training data to ensure validation split represents the spread of the training set
    # validation split takes data from the end of the dataset, so the data cannot be in chronological order
    regional_train_dfs[region_name] = shuffle(regional_train_dfs[region_name], random_state=42)

    # Reset index
    regional_train_dfs[region_name] = regional_train_dfs[region_name].reset_index(drop=True)

    # Print the lengths of the split datasets
    print(region_name,  'Train Dataset: ', len(regional_train_dfs[region_name]))
    print(region_name, 'Test Dataset: ', len(regional_test_dfs[region_name]))
    print()

North Train Dataset:  72349
North Test Dataset:  1721

South Train Dataset:  21517
South Test Dataset:  580

East Train Dataset:  44676
East Test Dataset:  1175

West Train Dataset:  51692
West Test Dataset:  1365

Central Train Dataset:  33427
Central Test Dataset:  803



### Encoding Categorical Features
For 'month_sold' feature:
- Perform cyclical encoding on the new column, which splits the column into two columns using sine and cosine:
    - sin(2π * month/12)
    - cos(2π * month/12)
- This models the feature in a cycle, meaning that 12 is as close to 1 as is to 11. This preserves the meaning of the month column.

For the rest of the categorical features:
- Perform one-hot-encoding

In [5]:
categorical_cols = ['town', 'flat_type', 'storey_range', 'flat_model']

# Initialise dictionaries to store encoded split dataframes
regional_train_dfs_encoded = {}
regional_test_dfs_encoded = {}

# One-hot-encode each dataset
for region_name in regions.keys():
    train_df = regional_train_dfs[region_name].copy()
    test_df = regional_test_dfs[region_name].copy()
    train_df_encoded = regional_train_dfs[region_name].copy()
    test_df_encoded = regional_test_dfs[region_name].copy()

    # Apply cyclical encoding to 'month_sold' feature
    train_df_encoded['month_sold_sin'] = np.sin(2 * np.pi * train_df['month_sold']/12)
    train_df_encoded['month_sold_cos'] = np.cos(2 * np.pi * train_df['month_sold']/12)
    # Drop original month sold feature
    train_df_encoded = train_df_encoded.drop(['month_sold'], axis=1)

    test_df_encoded['month_sold_sin'] = np.sin(2 * np.pi * test_df['month_sold']/12)
    test_df_encoded['month_sold_cos'] = np.cos(2 * np.pi * test_df['month_sold']/12)
    # Drop original month sold feature
    test_df_encoded = test_df_encoded.drop(['month_sold'], axis=1)

    # Apply one-hot-encoding to the training dataframe
    train_df_encoded = pd.get_dummies(train_df_encoded, columns=categorical_cols, drop_first=False, dtype=int)

    # Apply one-hot-encoding to the testing dataframe
    test_df_encoded = pd.get_dummies(test_df_encoded, columns=categorical_cols, drop_first=False, dtype=int)

    # Align columns for consistent feature sets between training and testing
    # This ensures that if a category is present in training but not testing or vice versa,
    # the dataFrames still have all columns, with missing ones filled with 0s.
    aligned_columns = list(train_df_encoded.columns)
    test_df_encoded = test_df_encoded.reindex(columns=aligned_columns, fill_value=0)

    # Store the encoded dataframes
    regional_train_dfs_encoded[region_name] = train_df_encoded
    regional_test_dfs_encoded[region_name] = test_df_encoded

print('Encoded regional dataframes created.')

Encoded regional dataframes created.


### Scaling Continuous Features

Scale the continuous features using StandardScaler(), ensuring the test sets are transformed using the same scaler used for the training sets.

In [6]:
# Define the continuous features to scale
continuous_cols = ['floor_area_sqm', 'resale_price', 'months_since_jan_2017', 'remaining_lease_years']

# Initialise dictionaries to store the final scaled split dataframes and the scalers
regional_train_dfs_final = {}
regional_test_dfs_final = {}
regional_scalers = {} # New dictionary to store scalers

for region_name in regions.keys():
    train_df_encoded = regional_train_dfs_encoded[region_name].copy()
    test_df_encoded = regional_test_dfs_encoded[region_name].copy()

    # Initialise a new StandardScaler for each dataframe
    scaler = StandardScaler()

    # Fit the scaler on the training data's continuous features and transform them
    train_df_encoded[continuous_cols] = scaler.fit_transform(train_df_encoded[continuous_cols])

    # Transform the testing data's continuous features using the fitted scaler
    test_df_encoded[continuous_cols] = scaler.transform(test_df_encoded[continuous_cols])

    # Store the final dataframes and the scaler
    regional_train_dfs_final[region_name] = train_df_encoded
    regional_test_dfs_final[region_name] = test_df_encoded
    regional_scalers[region_name] = scaler # Save the scaler

print('Scaled regional dataframes created.')

Scaled regional dataframes created.


# Local Single-Task Approach

This refers to dividing the dataset into groups in order to focus on predictions for a specific task. Multiple ML models are trained and tested. Standard error metric values are used to evaluate the separate models, but the weighted error metric value must be calculated before comparison can be done.

Task selection: We have decided to use 'Town' feature to divide the dataset into regions for this baseline. The model we will be using is a standard Multi-Layer Perceptron.

---

Neural networks are naturally stochastic, meaning certain operations are always random, such as:
- Model Weight Initialisation: Initial weights assigned to neurons are random.
- Dropout Layers: A set percentage but neurons chosen are random.
- Batches trained: The order is random.

Because of this, the model's results are different each time. By using a fixed random seed, the operations will be randomised in the same way each time. This will result in improved reproducibility (though results will still vary slightly).

---

The performance metrics used are:
1. Mean Squared Error (MSE) - Used for the loss function. MSE prioritises reducing the difference between predicted and actual values, so it this the best metric for house price prediction.
3. Root Mean Squared Error (RMSE) - Used as a performance metric to allow the results to be easily interpretable, it presents the loss in the same units as the target. However since the target variable was scaled, RMSE must be unscaled to be interpretable.
4. R-squared - Used as a performance metric to allow the results to be easily interpretable, it presents the percentage of variance explained by the model.

### Initial 5 Models

The neural network architecture and parameters used will be the same as the one used for the first baseline to ensure consistency, however tuning will differ as it will aim to maximise the performance of the 5 separate models.

In [7]:
# Initialise dictionaries to store X_train, y_train, X_test, y_test for each split
regional_X_train = {}
regional_y_train = {}
regional_X_test = {}
regional_y_test = {}

for region_name in regions.keys():
    # Get the final scaled dataframes for the current region
    train_df = regional_train_dfs_final[region_name]
    test_df = regional_test_dfs_final[region_name]

    # Split features (X) and target (y) for training set
    regional_X_train[region_name] = train_df.drop(columns=['resale_price'])
    regional_y_train[region_name] = train_df['resale_price']

    # Split features (X) and target (y) for testing set
    regional_X_test[region_name] = test_df.drop(columns=['resale_price'])
    regional_y_test[region_name] = test_df['resale_price']

    # Shuffle the training data to ensure the data is not in chronological order
    regional_X_train[region_name], regional_y_train[region_name] = shuffle(regional_X_train[region_name], regional_y_train[region_name], random_state=42)

    # Reset index
    regional_X_train[region_name] = regional_X_train[region_name].reset_index(drop=True)
    regional_y_train[region_name] = regional_y_train[region_name].reset_index(drop=True)

print('5 split X and y datasets created.')

5 split X and y datasets created.


In [8]:
# Dictionaries to store best model results
regional_best_batch_size = {}
regional_best_MSE = {}
regional_best_model_r2 = {}
regional_best_model_y_pred = {}

# Loop through the 5 datasets in the model
for region_name in regions.keys():
    # Define the current X_train, y_train, X_test, y_test
    X_train = regional_X_train[region_name]
    y_train = regional_y_train[region_name]
    X_test = regional_X_test[region_name]
    y_test = regional_y_test[region_name]

    # Define the neural network model
    model = keras.Sequential([
      keras.Input(shape=(X_train.shape[1],)),
      keras.layers.Dense(128, activation='relu'),
      keras.layers.Dropout(0.2), # Add dropout for regularisation
      keras.layers.Dense(64, activation='relu'),
      keras.layers.Dropout(0.2), # Add dropout for regularisation
      keras.layers.Dense(32, activation='relu'),
      keras.layers.Dense(1, activation='linear') # Output layer for regression
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['r2_score'])

    # Create checkpoint callback by setting Earlystopping's patience value to more than the total epochs
    Checkpoint = EarlyStopping(monitor='val_loss', patience=500, restore_best_weights=True)

    # Train the model
    history = model.fit(
        X_train,
        y_train,
        epochs=30,
        batch_size=32,
        validation_split=0.2,
        callbacks=[Checkpoint],
        verbose=0
    )

    # Evaluate the model on the test set
    loss, r2 = model.evaluate(X_test, y_test, verbose=0)

    # Make predictions on the test set
    y_pred = model.predict(X_test).flatten()

    # Calculate additional metrics
    mse = loss
    rmse = np.sqrt(mse)

    regional_best_batch_size[region_name] = 32 # Batch size of the initial model
    regional_best_MSE[region_name] = mse # MSE of the initial model
    regional_best_model_r2[region_name] = r2 # R-squared of the initial model
    regional_best_model_y_pred[region_name] = y_pred # Predictions of the initial model

    print('Neural Network Results for Region: ', region_name)
    print('Mean Squared Error (MSE): ', mse)
    print('Root Mean Squared Error (RMSE): ', rmse)
    print('R-squared: ', round(r2 * 100, 2))
    print()

54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Neural Network Results for Region:  North
Mean Squared Error (MSE):  0.05234646052122116
Root Mean Squared Error (RMSE):  0.22879348880862227
R-squared:  92.07

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Neural Network Results for Region:  South
Mean Squared Error (MSE):  0.06071791052818298
Root Mean Squared Error (RMSE):  0.24641004550988377
R-squared:  94.15

37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Neural Network Results for Region:  East
Mean Squared Error (MSE):  0.04279015213251114
Root Mean Squared Error (RMSE):  0.20685780655443278
R-squared:  93.86

43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Neural Network Results for Region:  West
Mean Squared Error (MSE):  0.0524924136698246
Root Mean Squared Error (RMSE):  0.22911222942004777
R-squared:  92.93

26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Neural Network Results for Region:  Central
Mean Squared Error (MSE):  0.040149398148059845
Root Mean Squared Error (RMSE):  0.20037314727293137
R-squared:  95.62



### Tuning

The initial models' performance are very good, with some performing slightly better than others. We will try tuning the batch size using a larger value.

In [9]:
# Loop through the 5 datasets in the model
for region_name in regions.keys():
    # Define the current X_train, y_train, X_test, y_test
    X_train = regional_X_train[region_name]
    y_train = regional_y_train[region_name]
    X_test = regional_X_test[region_name]
    y_test = regional_y_test[region_name]

    # Define the neural network model
    model = keras.Sequential([
      keras.Input(shape=(X_train.shape[1],)),
      keras.layers.Dense(128, activation='relu'),
      keras.layers.Dropout(0.2), # Add dropout for regularisation
      keras.layers.Dense(64, activation='relu'),
      keras.layers.Dropout(0.2), # Add dropout for regularisation
      keras.layers.Dense(32, activation='relu'),
      keras.layers.Dense(1, activation='linear') # Output layer for regression
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['r2_score'])

    # Create checkpoint callback by setting Earlystopping's patience value to more than the total epochs
    Checkpoint = EarlyStopping(monitor='val_loss', patience=500, restore_best_weights=True)

    # Train the model
    history = model.fit(
        X_train,
        y_train,
        epochs=30,
        batch_size=64,
        validation_split=0.2,
        callbacks=[Checkpoint],
        verbose=0
    )

    # Evaluate the model on the test set
    loss, r2 = model.evaluate(X_test, y_test, verbose=0)

    # Make predictions on the test set
    y_pred = model.predict(X_test).flatten()

    mse = loss
    print('Mean Squared Error (MSE): ', mse)
    print()

    print('Neural Network Tuning Results for Region: ', region_name)
    if mse < regional_best_MSE[region_name]:
      regional_best_MSE[region_name] = mse
      regional_best_batch_size[region_name] = 64
      regional_best_model_r2[region_name] = r2
      regional_best_model_y_pred[region_name] = y_pred
      print('New best model found with MSE: ', regional_best_MSE[region_name])
      print()
    else:
      print('No new best model found.')
      print()

54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Mean Squared Error (MSE):  0.05745325982570648

Neural Network Tuning Results for Region:  North
No new best model found.

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Mean Squared Error (MSE):  0.057712651789188385

Neural Network Tuning Results for Region:  South
New best model found with MSE:  0.057712651789188385

37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Mean Squared Error (MSE):  0.044095057994127274

Neural Network Tuning Results for Region:  East
No new best model found.

43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Mean Squared Error (MSE):  0.05682240426540375

Neural Network Tuning Results for Region:  West
No new best model found.

26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Mean Squared Error (MSE):  0.0486469529569149

Neural Network Tuning Results for Region:  Central
No new best model found.



### Final 5 Models

For each of the 5 models, use the learning rates that resulted in their best performance. Then calculate weighted results of the 5 models.

In [10]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Initialise a dictionary to store the MSE values in dollars
regional_MSE_dollars = {}

# Get the index of 'resale_price' in continuous_cols
resale_price_idx = continuous_cols.index('resale_price')

# Loop through the 5 datasets in the model
for region_name in regions.keys():
    # Get the predictions from the best model of the current region
    y_pred = regional_best_model_y_pred[region_name]

    # Get the actual values from the test set of the current region
    y_test = regional_y_test[region_name]

    # Retrieve the scaler used for this region from the dictionary
    scaler = regional_scalers[region_name]

    # Get the mean and standard deviation for 'resale_price' from the retrieved scaler
    original_resale_price_mean = scaler.mean_[resale_price_idx]
    original_resale_price_std = scaler.scale_[resale_price_idx]

    # StandardScaler takes the values and subtracts the mean before dividing by the standard deviation
    # To reverse scaling, multiply the scaled values by the standard deviation and then add the mean
    y_pred_log_regional = (y_pred * original_resale_price_std) + original_resale_price_mean
    y_test_log_regional = (y_test.values * original_resale_price_std) + original_resale_price_mean

    # Reverse log transformation
    y_pred_dollars_regional = np.expm1(y_pred_log_regional)
    y_test_dollars_regional = np.expm1(y_test_log_regional)

    # Calculate RMSE in dollars with the newly unscaled values
    mse_temp = mean_squared_error(y_test_dollars_regional, y_pred_dollars_regional)
    rmse = np.sqrt(mse_temp)
    regional_MSE_dollars[region_name] = mse_temp # Store MSE in dollars

    print('Final Neural Network Results for Region: ', region_name)
    print('Best batch size: ', regional_best_batch_size[region_name])
    print('Mean Squared Error (MSE): ', regional_best_MSE[region_name])
    print('Root Mean Squared Error (RMSE): ', np.sqrt(regional_best_MSE[region_name]))
    print('RMSE in dollars: ', rmse)
    print('R-squared: ', round(regional_best_model_r2[region_name] * 100, 2))
    print()

Final Neural Network Results for Region:  North
Best batch size:  32
Mean Squared Error (MSE):  0.05234646052122116
Root Mean Squared Error (RMSE):  0.22879348880862227
RMSE in dollars:  43454.151664304874
R-squared:  92.07

Final Neural Network Results for Region:  South
Best batch size:  64
Mean Squared Error (MSE):  0.057712651789188385
Root Mean Squared Error (RMSE):  0.2402345765896083
RMSE in dollars:  84860.9317532831
R-squared:  94.44

Final Neural Network Results for Region:  East
Best batch size:  32
Mean Squared Error (MSE):  0.04279015213251114
Root Mean Squared Error (RMSE):  0.20685780655443278
RMSE in dollars:  47851.209619505746
R-squared:  93.86

Final Neural Network Results for Region:  West
Best batch size:  32
Mean Squared Error (MSE):  0.0524924136698246
Root Mean Squared Error (RMSE):  0.22911222942004777
RMSE in dollars:  50673.54102448314
R-squared:  92.93

Final Neural Network Results for Region:  Central
Best batch size:  32
Mean Squared Error (MSE):  0.040149

In [11]:
# Calculate weighted performance metrics for comparison
# using Weighted_metric = sum(Region metric * (Test samples in region/Total samples in test set))

Total_test_samples = sum(len(regional_test_dfs_final[region_name]) for region_name in regions.keys())
weighted_mse = 0
weighted_mse_dollars = 0 # Temporary variable to store weighted MSE in dollars
weighted_r2 = 0

# Calculate the weighted metrics
for region_name in regions.keys():
  num_samples_in_region = len(regional_test_dfs_final[region_name])
  weight = num_samples_in_region / Total_test_samples

  weighted_mse = weighted_mse + (regional_best_MSE[region_name] * weight)
  weighted_mse_dollars = weighted_mse_dollars + (regional_MSE_dollars[region_name] * weight) # Use stored MSE in dollars
  weighted_r2 = weighted_r2 + (regional_best_model_r2[region_name] * weight)

# For weighted RMSE in dollars, take the square root of the weighted MSE in dollars
weighted_rmse_dollars = np.sqrt(weighted_mse_dollars)

print('Local Single-Task Approach Final Results: ')
print('Mean Squared Error (MSE): ', weighted_mse)
print('Root Mean Squared Error (RMSE): ', np.sqrt(weighted_mse))
print('RMSE in dollars: ', weighted_rmse_dollars)
print('R-squared: ', round(weighted_r2 * 100, 2))

Local Single-Task Approach Final Results: 
Mean Squared Error (MSE):  0.04920838708764247
Root Mean Squared Error (RMSE):  0.2218296352781622
RMSE in dollars:  56411.259750021796
R-squared:  93.4
